# M1 · Vectors as Meaning — companion notebook

> **Play with this.** This notebook is a *demonstration, not an assessment* — the module's real assessment is its problem set, done on paper. Everything here builds intuitions that are easier to reach by running code than by hand: what normalisation does to real count vectors, why cosine beats Euclidean distance for text, and the strange geometry of high-dimensional spaces.

Companion to the **Vectors as Meaning** module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

## 1 · Your data was always geometric

A survey respondent who answered a ten-question battery (say, agreement on a 1–7 scale) **is** a point in ten-dimensional space. Below, three synthetic respondents: two with similar politics, one different. No embedding model in sight — the geometry is already in the data.

In [ ]:
resp_a = np.array([6, 5, 7, 6, 2, 1, 6, 5, 7, 6])   # respondent A
resp_b = np.array([5, 6, 6, 7, 1, 2, 5, 6, 6, 5])   # similar views to A
resp_c = np.array([2, 1, 2, 1, 6, 7, 2, 1, 2, 2])   # very different views

def dist(u, v):
    return np.linalg.norm(u - v)

print(f"distance A–B (similar politics):  {dist(resp_a, resp_b):.2f}")
print(f"distance A–C (opposed politics):  {dist(resp_a, resp_c):.2f}")

Small distance = near-agreement on every question at once. And the *direction* from one respondent to another says **which questions** account for the disagreement — directions in these spaces carry meaning.

## 2 · Norms, and what normalisation throws away

For text represented as word counts, the norm mostly measures **document length**. Triple a document and its count vector triples: same direction, three times the reach. Normalisation deletes exactly that.

In [ ]:
doc = np.array([2, 1, 4])          # counts of three vocabulary words
doc_3x = 3 * doc                    # the same text, three times over

print(f"norm of doc:    {np.linalg.norm(doc):.3f}")
print(f"norm of 3x doc: {np.linalg.norm(doc_3x):.3f}   (three times larger)")

unit = doc / np.linalg.norm(doc)
unit_3x = doc_3x / np.linalg.norm(doc_3x)
print(f"\nnormalised doc:    {np.round(unit, 4)}")
print(f"normalised 3x doc: {np.round(unit_3x, 4)}   (identical — length is gone)")
print(f"unit length check: {np.linalg.norm(unit):.4f}")

## 3 · One number, three readings

The dot product can be computed from coordinates (what machines do) or from lengths and the angle (what it means). Same number, to the last decimal.

In [ ]:
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])

algebraic = np.dot(a, b)                     # sum of coordinate products

theta = np.arccos(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
geometric = np.linalg.norm(a) * np.linalg.norm(b) * np.cos(theta)

print(f"algebraic reading:  {algebraic:.6f}")
print(f"geometric reading:  {geometric:.6f}")
print(f"angle between them: {np.degrees(theta):.1f} degrees")

## 4 · Cosine vs Euclidean, on documents

A tweet and a long report about the *same topic*: Euclidean distance says they are far apart (it is measuring length); cosine says they are nearly identical (it is measuring direction). A tweet about a *different topic* shows the reverse pattern.

In [ ]:
# Count vectors over the same vocabulary
tweet_climate  = np.array([2, 1, 0, 1, 0, 0])      # short text on climate
report_climate = np.array([38, 21, 3, 17, 2, 1])   # long report on climate
tweet_sports   = np.array([0, 0, 2, 0, 1, 2])      # short text on sport

def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

print("tweet_climate vs report_climate  (same topic, very different length)")
print(f"  euclidean: {dist(tweet_climate, report_climate):7.2f}   cosine: {cosine(tweet_climate, report_climate):.3f}")
print("tweet_climate vs tweet_sports    (different topic, same length)")
print(f"  euclidean: {dist(tweet_climate, tweet_sports):7.2f}   cosine: {cosine(tweet_climate, tweet_sports):.3f}")

Euclidean distance ranks the *same-topic* pair as far more different than the *different-topic* pair — exactly backwards for a question about content. Cosine gets it right. This is why text pipelines, from tf-idf to modern embeddings, compare by angle.

## 5 · The high-dimensional surprise

Problem 6 of the module's problem set, run live: dot products of **random** unit vectors concentrate near zero as dimension grows — in 768 dimensions, almost everything is almost orthogonal to almost everything else. Embeddings are useful precisely because they are *not* random: training packs related meanings into a thin, structured region of a mostly-empty space.

In [ ]:
def random_unit_dots(d, n=4000):
    u = rng.standard_normal((n, d))
    v = rng.standard_normal((n, d))
    u /= np.linalg.norm(u, axis=1, keepdims=True)
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    return np.sum(u * v, axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
for d in [2, 32, 768]:
    ax.hist(random_unit_dots(d), bins=80, density=True, alpha=0.55, label=f"d = {d}")
ax.set_xlabel("dot product of two random unit vectors")
ax.set_ylabel("density")
ax.set_title("Random unit vectors concentrate near orthogonality as dimension grows")
ax.legend()
plt.tight_layout()
plt.show()

for d in [2, 32, 768]:
    print(f"d = {d:4d}:  std of dot product = {np.std(random_unit_dots(d)):.3f}   (about 1/sqrt(d) = {1/np.sqrt(d):.3f})")

## 6 · Where this lands: attention scores

An attention score is a dot product between a query vector and a key vector. One preview of *why the scaling factor exists* — the module's deliberately open question: watch what happens to the spread of raw dot products as dimension grows, and what dividing by √d does to it. The full explanation needs variance machinery and belongs to M7.

In [ ]:
for d in [8, 64, 512]:
    q = rng.standard_normal((2000, d))
    k = rng.standard_normal((2000, d))
    raw = np.sum(q * k, axis=1)
    scaled = raw / np.sqrt(d)
    print(f"d = {d:4d}:  std of q·k = {np.std(raw):6.2f}    std of q·k/sqrt(d) = {np.std(scaled):.2f}")

The raw spread grows with dimension; the scaled spread stays put at 1. *Why* the growth is exactly √d — and why a large spread would break the softmax that comes next — is M7's story. Hold the question.

---

**Next:** M2 · Matrices as Transformations — where a matrix stops being a table of numbers and becomes a function.